In [1]:
import pandas as pd 
import numpy as np 
import tensorflow as tf 

In [2]:
df  = pd.read_csv("realistic_balanced_chat_data.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   conversation_id  500 non-null    int64 
 1   patient_message  500 non-null    object
 2   doctor_response  500 non-null    object
 3   intent           500 non-null    object
 4   sentiment        500 non-null    object
 5   summary          500 non-null    object
 6   key_points       500 non-null    object
dtypes: int64(1), object(6)
memory usage: 27.5+ KB


In [3]:
import nltk
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to C:\Users\arfit/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arfit/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\arfit/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [10]:
import re 
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

stop_words = set(stopwords.words("english"))
ps = PorterStemmer()

In [13]:
import string
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [8]:
req_df = df[["patient_message","sentiment"]]
req_df['label'] = le.fit_transform(req_df['sentiment'])

C:\Users\arfit\AppData\Local\Temp\ipykernel_25812\4089496017.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  req_df['label'] = le.fit_transform(req_df['sentiment'])


In [9]:
req_df

,patient_message,sentiment,label
0,I've been waiting 45 minutes past my scheduled...,formal,2
1,What COVID safety measures are you currently f...,neutral,5
2,The antidepressant dosage doesn't seem effecti...,friendly,3
3,Your online portal isn't working and I can't a...,negative,4
4,I'd like to make a same-day urgent appointment...,neutral,5
...,...,...,...
495,My child needs a sports physical - do you have...,neutral,5
496,The doctor didn't explain my diagnosis clearly...,apologetic,0
497,There's sharp pain in my lower right abdomen t...,urgent,7
498,Do you accept my new insurance plan?,friendly,3


In [14]:
def tokeniser(obj):
    obj = obj.lower()
    obj = nltk.word_tokenize(obj)
    res = [ word for word in obj  if word.isalnum()]
    obj = res 
    res = [i  for i in obj  if i not in stopwords.words('english') and  i not in string.punctuation]
    obj = res
    res = [ ps.stem(i)  for i in obj ]
    return " ".join(res)

In [15]:
tokeniser(" This i will meet you in the luch break eactly in 12pm at the stree! like we all ways use to meets i hope you don't forget")

'meet luch break eactli 12pm stree like way use meet hope forget'

In [16]:
req_df['preprocessed_PM'] = req_df['patient_message'].apply(tokeniser)

C:\Users\arfit\AppData\Local\Temp\ipykernel_25812\1132334796.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  req_df['preprocessed_PM'] = req_df['patient_message'].apply(tokeniser)


In [17]:
req_df

,patient_message,sentiment,label,preprocessed_PM
0,I've been waiting 45 minutes past my scheduled...,formal,2,wait 45 minut past schedul appoint time
1,What COVID safety measures are you currently f...,neutral,5,covid safeti measur current follow
2,The antidepressant dosage doesn't seem effecti...,friendly,3,antidepress dosag seem effect anymor adjust
3,Your online portal isn't working and I can't a...,negative,4,onlin portal work ca access medic record
4,I'd like to make a same-day urgent appointment...,neutral,5,like make urgent appoint possibl
...,...,...,...,...
495,My child needs a sports physical - do you have...,neutral,5,child need sport physic pediatr appoint tomorrow
496,The doctor didn't explain my diagnosis clearly...,apologetic,0,doctor explain diagnosi clearli mani unansw qu...
497,There's sharp pain in my lower right abdomen t...,urgent,7,sharp pain lower right abdomen come goe
498,Do you accept my new insurance plan?,friendly,3,accept new insur plan


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [21]:
x = tfidf.fit_transform(req_df['preprocessed_PM']).toarray()

In [23]:
x.shape

(500, 213)

In [24]:
y = req_df['label']

In [25]:
from sklearn.model_selection import train_test_split

In [27]:
x_train,x_test,y_train,y_test = train_test_split(x,y, test_size=0.2, random_state = 42)

In [29]:
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score

In [30]:
gnb = GaussianNB ()
mnb = MultinomialNB ()
bnb = BernoulliNB ()

In [35]:
gnb.fit(x_train,y_train)
y_pred = gnb.predict(x_test)
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(precision_score(y_test, y_pred, average='macro'))

0.08
[[ 5  0  0  1  1  0  0  0]
 [ 8  0  0  1  1  0  0  0]
 [ 8  1  2  0  3  0  0  0]
 [ 2  0  5  0  0  0  0  0]
 [ 5  4  4  1  1  0  0  0]
 [11  3  6  1  3  0  0  0]
 [ 7  1  3  0  2  0  0  0]
 [ 6  3  0  1  0  0  0  0]]
0.03588286713286713


c:\works\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [36]:
mnb.fit(x_train,y_train)
y_pred = gnb.predict(x_test)
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(precision_score(y_test, y_pred, average='macro'))

0.08
[[ 5  0  0  1  1  0  0  0]
 [ 8  0  0  1  1  0  0  0]
 [ 8  1  2  0  3  0  0  0]
 [ 2  0  5  0  0  0  0  0]
 [ 5  4  4  1  1  0  0  0]
 [11  3  6  1  3  0  0  0]
 [ 7  1  3  0  2  0  0  0]
 [ 6  3  0  1  0  0  0  0]]
0.03588286713286713


c:\works\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [37]:
bnb.fit(x_train,y_train)
y_pred = gnb.predict(x_test)
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(precision_score(y_test, y_pred, average='macro'))

0.08
[[ 5  0  0  1  1  0  0  0]
 [ 8  0  0  1  1  0  0  0]
 [ 8  1  2  0  3  0  0  0]
 [ 2  0  5  0  0  0  0  0]
 [ 5  4  4  1  1  0  0  0]
 [11  3  6  1  3  0  0  0]
 [ 7  1  3  0  2  0  0  0]
 [ 6  3  0  1  0  0  0  0]]
0.03588286713286713


c:\works\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [38]:

from sklearn.linear_model import LogisticRegression 
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

In [46]:
svc = SVC(kernel = 'linear',C=1, probability=True)
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth = 5)
lrc = LogisticRegression(solver = 'liblinear' , penalty ='l1')
rfc = RandomForestClassifier(n_estimators = 50 , random_state = 2)
abc = AdaBoostClassifier(n_estimators = 50 , random_state = 2)
bc  = BaggingClassifier(n_estimators = 50 , random_state = 2)
etc = ExtraTreesClassifier(n_estimators = 50 , random_state = 2)
gbdt = GradientBoostingClassifier(n_estimators = 50 , random_state = 2)
xgb = XGBClassifier(n_estimators = 50 , random_state = 2)

In [47]:
models = {
    'SVC' : svc,
     'KN' : knc,
     'NB' : mnb,
     "DT" : dtc,
     "LR" : lrc,
     'RF' : rfc,
     'AB' : abc,
     'BC' : bc,
     'ETC': etc,
    'GBDT': gbdt,
     'XGB': xgb
}

In [48]:
def model_trainer(models, x_train,y_train,x_test,y_test):
    models.fit(x_train,y_train)
    y_prediction = models.predict(x_test)
    accuracy = accuracy_score(y_test,y_prediction)
    precision = precision_score(y_test,y_prediction,average='macro')
    return accuracy,precision

In [49]:
accuracy_scores = []
precision_scores = []

for name , model in models.items():
    current_accuracy,current_precision = model_trainer(model,x_train,y_train,x_test,y_test)
    print('For ', name)
    print('Accuracy - ' , current_accuracy)
    print('Precision - ' , current_precision)

    accuracy_scores.append(current_accuracy)
    precision_scores.append(current_precision)

c:\works\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\works\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\works\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\works\.venv\Lib\site-packages\sklearn\metrics\_classification.py:15

For  SVC
Accuracy -  0.18
Precision -  0.1427255318970435
For  KN
Accuracy -  0.15
Precision -  0.17118055555555556
For  NB
Accuracy -  0.17
Precision -  0.11147553189704353
For  DT
Accuracy -  0.22
Precision -  0.03125
For  LR
Accuracy -  0.23
Precision -  0.030913978494623656
For  RF
Accuracy -  0.19
Precision -  0.18125
For  AB
Accuracy -  0.2
Precision -  0.14069940476190476
For  BC
Accuracy -  0.19
Precision -  0.18125
For  ETC
Accuracy -  0.15
Precision -  0.12673139768728003


c:\works\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


For  GBDT
Accuracy -  0.2
Precision -  0.15347222222222223
For  XGB
Accuracy -  0.16
Precision -  0.1304563492063492
